<a href="https://colab.research.google.com/github/Rhea-Shah23/indiancine.ma-scrape-curate/blob/main/MACS207_Indiancine_ma_Scrape_and_Curate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!playwright install-deps

In [ ]:
!pip install playwright

In [ ]:
!playwright install

In [ ]:
# Use Playwright to scrape dynamic content from Indian Cine.ma
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
from imdb import IMDb
import re

# Install and apply nest_asyncio to allow nested event loops in Jupyter Notebook
!pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

# Initialize IMDb API
ia = IMDb()

# Function to scrape data using Playwright
async def scrape_with_playwright():
    movies = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)  # Set headless=False to see the browser
        page = await browser.new_page()
        await page.goto("https://indiancine.ma/grid/year")

        # Wait for the page to load the movie elements
        await page.wait_for_selector('.OxElement')

        await asyncio.sleep(5)

        # Extract movie elements
        movie_elements = await page.query_selector_all('.OxText')
        print(movie_elements)
        for movie in movie_elements:
            try:
                title_element = await movie.query_selector('.OxTarget')
                if title_element:
                    full_text = await title_element.text_content()
                    match = re.match(r"^(.*?)(\d{4})$", full_text.strip())
                    if match:
                        title = match.group(1).strip()
                        year = match.group(2).strip()
                    else:
                        title = full_text.strip()
                        year = "N/A"

                    # Search for the movie on IMDb
                    search_results = ia.search_movie(title)
                    if search_results:
                        imdb_id = search_results[0].movieID
                    else:
                        imdb_id = 'N/A'

                    movies.append({"Title": title, "Year": year, "IMDbID": imdb_id})
            except Exception as e:
                print(f"Error extracting data for a movie: {e}")

        await browser.close()

    # Save the extracted data into a CSV file
    df = pd.DataFrame(movies)
    display(df)
    # output_file = "indian_cinema_movies_with_imdb_playwright.csv"
    # df.to_csv(output_file, index=False)
    # print(f"Data saved to {output_file}")

# Run the function
await scrape_with_playwright()